# LeetCode 2126 — Destroying Asteroids bằng Python

Link đề: https://leetcode.com/problems/destroying-asteroids/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def asteroidsDestroyed(self, mass: int, asteroids: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài **tham lam theo thứ tự sắp xếp**: "được chọn thứ tự tùy ý" + "càng ăn càng mạnh" ⇒ ăn từ nhỏ tới lớn.
2. Chứng minh tính đúng của tham lam bằng **lập luận đổi chỗ** (exchange argument) và bằng **phản chứng**.
3. So sánh: vét cạn mọi hoán vị (O(n!·n)), mô phỏng "chọn thiên thạch nào đó phá được" (O(n²)), sắp xếp (O(n log n)), đếm phân phối (O(n + V)).
4. Biết các tối ưu thực dụng: **dừng sớm** khi khối lượng đã ≥ thiên thạch lớn nhất.
5. Mở rộng: tìm **khối lượng ban đầu nhỏ nhất** cần thiết, đếm số thiên thạch phá được tối đa.

## 2. Đề bài & ví dụ

Hành tinh có khối lượng ban đầu `mass`. Mảng `asteroids` chứa khối lượng các thiên thạch.

Bạn được sắp xếp cho hành tinh va chạm với các thiên thạch theo **thứ tự tùy ý**. Khi va chạm:
- nếu khối lượng hành tinh **≥** khối lượng thiên thạch: thiên thạch bị phá, hành tinh **nhận thêm** khối lượng của nó;
- ngược lại: hành tinh bị phá hủy.

Trả về `True` nếu có thể phá **tất cả** thiên thạch.

**Ví dụ 1:** `mass = 10, asteroids = [3,9,19,5,21]` → `True` (một thứ tự: 9, 19, 5, 3, 21: khối lượng 19 → 38 → 43 → 46 → 67).

**Ví dụ 2:** `mass = 5, asteroids = [4,9,23,4]` → `False` (ăn hết các thiên thạch khác thì được 5 + 4 + 9 + 4 = 22 < 23).

**Ràng buộc:** `1 ≤ mass ≤ 10⁵`, `1 ≤ asteroids.length ≤ 10⁵`, `1 ≤ asteroids[i] ≤ 10⁵`.

In [1]:
from typing import List
from itertools import permutations, accumulate
import random, time

examples = [
    (10, [3, 9, 19, 5, 21], True),
    (5, [4, 9, 23, 4], False),
]
def play(mass, order, verbose=False):
    for x in order:
        if mass < x:
            if verbose: print(f"   gặp {x} khi mass={mass} -> hành tinh bị phá")
            return False
        mass += x
        if verbose: print(f"   phá {x} -> mass={mass}")
    return True

print("Ví dụ 1, thứ tự của đề [9,19,5,3,21]:"); play(10, [9, 19, 5, 3, 21], True)
print("Ví dụ 1, thứ tự gốc [3,9,19,5,21]:"); play(10, [3, 9, 19, 5, 21], True)

Ví dụ 1, thứ tự của đề [9,19,5,3,21]:
   phá 9 -> mass=19
   phá 19 -> mass=38
   phá 5 -> mass=43
   phá 3 -> mass=46
   phá 21 -> mass=67
Ví dụ 1, thứ tự gốc [3,9,19,5,21]:
   phá 3 -> mass=13
   phá 9 -> mass=22
   phá 19 -> mass=41
   phá 5 -> mass=46
   phá 21 -> mass=67


True

## 3. Kiến thức nền

### 3.1 `sorted()` vs `list.sort()`

- `sorted(a)` trả về list **mới**, không đổi `a`.
- `a.sort()` sắp xếp **tại chỗ**, trả về `None`, làm thay đổi list của người gọi.

Cả hai đều O(n log n) (Timsort). Trên LeetCode sửa input không sao, nhưng khi viết hàm thư viện nên dùng `sorted` để tránh tác dụng phụ.

In [2]:
a = [3, 9, 19, 5, 21]
b = sorted(a)
print("a =", a, " b =", b)
r = a.sort()
print("a sau sort() =", a, " giá trị trả về:", r)

a = [3, 9, 19, 5, 21]  b = [3, 5, 9, 19, 21]
a sau sort() = [3, 5, 9, 19, 21]  giá trị trả về: None


### 3.2 Tổng tiền tố — "khối lượng sau khi ăn k thiên thạch đầu"

Nếu ăn theo thứ tự `x_0, x_1, …` thì trước khi gặp `x_k`, khối lượng là `mass + x_0 + … + x_{k−1}` = `mass + pre[k]`.

In [3]:
xs = [3, 5, 9, 19, 21]
pre = list(accumulate(xs, initial=0))
for k, x in enumerate(xs):
    print(f"trước khi gặp {x:>2}: mass = 10 + {pre[k]:>2} = {10 + pre[k]:>2}  {'>=' if 10 + pre[k] >= x else '<'} {x}")

trước khi gặp  3: mass = 10 +  0 = 10  >= 3
trước khi gặp  5: mass = 10 +  3 = 13  >= 5
trước khi gặp  9: mass = 10 +  8 = 18  >= 9
trước khi gặp 19: mass = 10 + 17 = 27  >= 19
trước khi gặp 21: mass = 10 + 36 = 46  >= 21


### 3.3 `itertools.permutations` — vét cạn thứ tự

Sinh mọi hoán vị; với `n` phần tử có `n!` hoán vị (8! = 40 320, 10! ≈ 3,6 triệu). Chỉ dùng để kiểm chứng trên input rất nhỏ.

In [4]:
print(list(permutations([1, 2, 3])))
print(sum(1 for _ in permutations(range(8))), "hoán vị cho n = 8")

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
40320 hoán vị cho n = 8


### 3.4 `all()` với generator — dừng sớm

`all(cond for ...)` dừng ngay khi gặp điều kiện sai đầu tiên — tiện cho kiểu "mọi bước đều thỏa".

In [5]:
def check(x):
    print("  kiểm tra", x); return x < 3
print(all(check(x) for x in [1, 2, 5, 0]))

  kiểm tra 1
  kiểm tra 2
  kiểm tra 5
False


## 4. Ý tưởng nền tảng / trực giác

Hãy hình dung hành tinh như một "quả cầu tuyết": càng lăn càng lớn. Bạn muốn khi gặp thiên thạch to nhất thì mình đã to nhất có thể.

- Ăn một thiên thạch **không bao giờ làm hành tinh nhỏ đi** (khối lượng chỉ tăng).
- Nếu hiện tại phá được thiên thạch `x` nhỏ thì phá nó luôn chẳng mất gì: các thiên thạch khác sau đó chỉ dễ hơn.
- Ngược lại, nếu cố "đâm" thiên thạch lớn trước khi đủ khối lượng thì thua ngay.

⇒ Chiến lược tự nhiên: **luôn ăn thiên thạch nhỏ nhất còn lại**, tức sắp xếp tăng dần rồi ăn lần lượt.

Tính tay ví dụ 2 với thứ tự tăng dần `[4, 4, 9, 23]` từ `mass = 5`:

| bước | thiên thạch | mass trước | phá được? | mass sau |
|---|---|---|---|---|
| 1 | 4 | 5 | 5 ≥ 4 ✔ | 9 |
| 2 | 4 | 9 | ✔ | 13 |
| 3 | 9 | 13 | ✔ | 22 |
| 4 | 23 | 22 | 22 < 23 ✘ | — |

Và vì 22 đã là khối lượng **lớn nhất có thể** trước khi gặp 23 (đã ăn hết mọi thứ khác), không thứ tự nào cứu được ⇒ `False`.

In [6]:
for mass, ast, exp in examples:
    print(f"mass={mass}, sắp xếp: {sorted(ast)}")
    print("   kết quả:", play(mass, sorted(ast), verbose=True), "(kỳ vọng", exp, ")")

mass=10, sắp xếp: [3, 5, 9, 19, 21]
   phá 3 -> mass=13
   phá 5 -> mass=18
   phá 9 -> mass=27
   phá 19 -> mass=46
   phá 21 -> mass=67
   kết quả: True (kỳ vọng True )
mass=5, sắp xếp: [4, 4, 9, 23]
   phá 4 -> mass=9
   phá 4 -> mass=13
   phá 9 -> mass=22
   gặp 23 khi mass=22 -> hành tinh bị phá
   kết quả: False (kỳ vọng False )


## 5. Cách 1 — Brute Force / baseline: thử mọi thứ tự

**Ý tưởng:** đề nói "thứ tự tùy ý" ⇒ thử mọi hoán vị, có một hoán vị thành công là `True`. Đúng theo định nghĩa nhưng O(n!·n).

In [7]:
def brute(mass, asteroids):
    return any(play(mass, order) for order in permutations(asteroids))

for mass, ast, exp in examples:
    got = brute(mass, ast)
    print(got, "OK" if got == exp else "SAI")

True OK
False OK


**Độ phức tạp:** O(n!·n) thời gian. n = 12 đã ~5,7 tỷ phép. Hoàn toàn không dùng được với n = 10⁵.

**Nút thắt:** rất nhiều hoán vị "tương đương". Câu hỏi đúng không phải "thứ tự nào?", mà "tại mỗi bước, nên ăn thiên thạch nào?" — và câu trả lời là bất kỳ thiên thạch nào đang phá được.

### 5.1 Baseline thứ hai: mô phỏng "ăn bất cứ thứ gì phá được" — O(n²)

Lặp: tìm một thiên thạch còn lại có khối lượng ≤ mass, ăn nó. Nếu không còn thiên thạch nào phá được mà vẫn còn thiên thạch ⇒ `False`.

In [8]:
def simulate_any(mass, asteroids):
    left = list(asteroids)
    while left:
        for idx, x in enumerate(left):
            if x <= mass:
                mass += x
                left.pop(idx)
                break
        else:                        # vòng for không break: không phá được ai
            return False
    return True

for mass, ast, exp in examples:
    print(simulate_any(mass, ast), "OK" if simulate_any(mass, ast) == exp else "SAI")

True OK
False OK


`for … else`: khối `else` chạy khi vòng `for` kết thúc **không** do `break` — tức không tìm thấy thiên thạch nào phá được.

**Độ phức tạp:** mỗi vòng `while` tìm tuyến tính O(n) và `pop` O(n) ⇒ O(n²). Với n = 10⁵: 10¹⁰ — vẫn chậm.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** nếu tồn tại thứ tự phá hết thì **thứ tự tăng dần** cũng phá hết. Do đó chỉ cần kiểm tra một thứ tự duy nhất.

**Lập luận đổi chỗ:** giả sử thứ tự thành công `O` có hai thiên thạch kề nhau `y` rồi `x` với `y > x`. Đổi thành `x` rồi `y`:
- Trước cặp này khối lượng là `M`. Trong `O`: `M ≥ y` (phá được `y`) ⇒ `M ≥ y > x` ⇒ phá được `x` trước.
- Sau khi phá `x`: `M + x ≥ M ≥ y` ⇒ phá được `y`.
- Sau cặp, khối lượng vẫn là `M + x + y` ⇒ phần sau không đổi.

Lặp lại việc đổi các cặp ngược thứ tự (như bubble sort) ta được thứ tự tăng dần, vẫn thành công. ∎

**Phát biểu tương đương:** sau khi sắp xếp tăng `a_0 ≤ a_1 ≤ … ≤ a_{n−1}`, đáp án `True` ⇔ với mọi `k`: `mass + a_0 + … + a_{k−1} ≥ a_k`.

**Quy tắc "nếu… thì…":**
- **Nếu** `mass ≥ a_k` **thì** ăn và `mass += a_k`.
- **Nếu** `mass < a_k` **thì** trả `False` ngay: mọi thiên thạch còn lại đều ≥ `a_k`, và `mass` hiện tại đã là lớn nhất có thể trước khi phải đối mặt với một thiên thạch ≥ `a_k`.
- **Nếu** `mass ≥ max(asteroids)` tại bất kỳ lúc nào **thì** trả `True` ngay (dừng sớm).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + quét (cách chuẩn)

In [9]:
def sort_scan(mass, asteroids):
    for x in sorted(asteroids):
        if mass < x:
            return False
        mass += x
    return True

for mass, ast, exp in examples:
    print(sort_scan(mass, ast), "OK" if sort_scan(mass, ast) == exp else "SAI")

True OK
False OK


Code nộp LeetCode cho cách 2 (sắp xếp tại chỗ như doocs — tiết kiệm bộ nhớ, chấp nhận sửa input):

In [10]:
class SolutionSort:
    def asteroidsDestroyed(self, mass: int, asteroids: List[int]) -> bool:
        asteroids.sort()
        for x in asteroids:
            if mass < x:
                return False
            mass += x
        return True

print([SolutionSort().asteroidsDestroyed(m, list(a)) for m, a, _ in examples])

[True, False]


**Độ phức tạp cách 2:** O(n log n) thời gian (sắp xếp), O(1) thêm nếu sort tại chỗ (Timsort cần O(n) bộ nhớ tạm trong worst case), O(n) nếu dùng `sorted`.

### 7.2 Cách 3 — Sắp xếp + dừng sớm khi đủ lớn

Tối ưu thực dụng: khi `mass ≥ max(asteroids)`, mọi thiên thạch còn lại đều phá được ⇒ `True`. Hữu ích khi `mass` ban đầu lớn: thậm chí khỏi sắp xếp.

In [11]:
def sort_early(mass, asteroids):
    mx = max(asteroids)
    if mass >= mx:
        return True                  # O(n), không cần sort
    for x in sorted(asteroids):
        if mass < x:
            return False
        mass += x
        if mass >= mx:
            return True
    return True

for mass, ast, exp in examples:
    print(sort_early(mass, ast), "OK" if sort_early(mass, ast) == exp else "SAI")

True OK
False OK


### 7.3 Cách 4 — Đếm phân phối (counting sort) O(n + V)

Vì `asteroids[i] ≤ V = 10⁵`, đếm số lần xuất hiện của mỗi khối lượng rồi duyệt khối lượng từ nhỏ tới lớn. Mỗi giá trị `v` với `cnt[v]` lần: cần `mass ≥ v` rồi `mass += v · cnt[v]` (sau khi ăn cái đầu tiên, các cái cùng giá trị chắc chắn phá được).

In [12]:
def counting(mass, asteroids):
    V = max(asteroids)
    cnt = [0] * (V + 1)
    for x in asteroids:
        cnt[x] += 1
    for v in range(1, V + 1):
        if cnt[v]:
            if mass < v:
                return False
            mass += v * cnt[v]
            if mass >= V:
                return True
    return True

for mass, ast, exp in examples:
    print(counting(mass, ast), "OK" if counting(mass, ast) == exp else "SAI")

True OK
False OK


**Độ phức tạp cách 4:** O(n + V) thời gian, O(V) bộ nhớ. Không nhanh hơn đáng kể so với `sorted` trong Python (sort viết bằng C) nhưng là ý tưởng quan trọng khi giá trị bị chặn.

### 7.4 Cách 5 — Công thức tiền tố (kiểm tra "mọi k")

Sau khi sắp xếp, đáp án là `all(mass + pre[k] >= a[k])`. Viết dạng một dòng với `accumulate`:

In [13]:
def prefix_formula(mass, asteroids):
    a = sorted(asteroids)
    pre = accumulate(a, initial=mass)           # mass, mass+a0, mass+a0+a1, ...
    return all(p >= x for p, x in zip(pre, a))

for mass, ast, exp in examples:
    print(prefix_formula(mass, ast), "OK" if prefix_formula(mass, ast) == exp else "SAI")

True OK
False OK


`accumulate(a, initial=mass)` sinh `mass, mass + a_0, mass + a_0 + a_1, …` — đúng "khối lượng trước khi gặp `a_k`". `zip` cắt theo list ngắn hơn nên phần tử cuối (tổng toàn bộ) bị bỏ qua tự nhiên.

## 8. TRACE từng bước

Trace cách 3 (sắp xếp + dừng sớm) trên ví dụ 1: `mass = 10`, sắp xếp `[3, 5, 9, 19, 21]`, `max = 21`.

| bước | x | mass trước | so sánh | mass sau | ≥ max? |
|---|---|---|---|---|---|
| 1 | 3 | 10 | 10 ≥ 3 | 13 | không |
| 2 | 5 | 13 | ✔ | 18 | không |
| 3 | 9 | 18 | ✔ | 27 | **có** ⇒ True |

In [14]:
def trace_sort_early(mass, asteroids):
    mx = max(asteroids); a = sorted(asteroids)
    print(f"mass={mass}, sắp xếp={a}, max={mx}")
    if mass >= mx:
        print("mass >= max ngay từ đầu -> True"); return True
    for step, x in enumerate(a, 1):
        if mass < x:
            print(f"bước {step}: x={x}, mass={mass} < x -> False"); return False
        mass += x
        flag = mass >= mx
        print(f"bước {step}: x={x} phá được -> mass={mass}{'  (>= max, dừng sớm: True)' if flag else ''}")
        if flag: return True
    return True

trace_sort_early(10, [3, 9, 19, 5, 21])
print()
trace_sort_early(5, [4, 9, 23, 4])

mass=10, sắp xếp=[3, 5, 9, 19, 21], max=21
bước 1: x=3 phá được -> mass=13
bước 2: x=5 phá được -> mass=18
bước 3: x=9 phá được -> mass=27  (>= max, dừng sớm: True)

mass=5, sắp xếp=[4, 4, 9, 23], max=23
bước 1: x=4 phá được -> mass=9
bước 2: x=4 phá được -> mass=13
bước 3: x=9 phá được -> mass=22
bước 4: x=23, mass=22 < x -> False


False

Trace mô phỏng O(n²) (cách 1b) để thấy nó chọn thiên thạch "đầu tiên phá được" — không nhất thiết là nhỏ nhất, nhưng vẫn đúng:

In [15]:
def trace_simulate_any(mass, asteroids):
    left = list(asteroids)
    while left:
        for idx, x in enumerate(left):
            if x <= mass:
                mass += x; left.pop(idx)
                print(f"ăn {x:>2} -> mass={mass:>3}, còn lại {left}")
                break
        else:
            print(f"không phá được ai trong {left} với mass={mass} -> False"); return False
    print("-> True"); return True

trace_simulate_any(10, [3, 9, 19, 5, 21])

ăn  3 -> mass= 13, còn lại [9, 19, 5, 21]
ăn  9 -> mass= 22, còn lại [19, 5, 21]
ăn 19 -> mass= 41, còn lại [5, 21]
ăn  5 -> mass= 46, còn lại [21]
ăn 21 -> mass= 67, còn lại []
-> True


True

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def asteroidsDestroyed(self, mass: int, asteroids: List[int]) -> bool:
        for x in sorted(asteroids):     # ăn từ nhỏ đến lớn
            if mass < x:
                return False            # mọi thiên thạch còn lại đều >= x
            mass += x
        return True

In [17]:
sol = Solution()
for mass, ast, exp in examples:
    got = sol.asteroidsDestroyed(mass, ast)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

True | kỳ vọng True | OK
False | kỳ vọng False | OK


## 10. Vì sao thuật toán đúng

Gọi `a_0 ≤ … ≤ a_{n−1}` là mảng sau khi sắp xếp.

**(1) Nếu thuật toán trả `True` thì đáp án là `True`:** thuật toán vừa đưa ra một thứ tự cụ thể (tăng dần) phá hết.

**(2) Nếu thuật toán trả `False` thì đáp án là `False`:** thuật toán dừng ở `a_k` với `M_k = mass + a_0 + … + a_{k−1} < a_k`. Xét **bất kỳ** thứ tự nào. Gọi `y` là thiên thạch đầu tiên trong thứ tự đó thuộc nhóm `{a_k, …, a_{n−1}}` (nhóm có `n − k` phần tử, đều ≥ `a_k`). Trước `y`, hành tinh chỉ ăn được các phần tử thuộc `{a_0, …, a_{k−1}}` ⇒ khối lượng ≤ `M_k < a_k ≤ y` ⇒ thua tại `y`. ∎

Lập luận (2) cho thấy rõ "ý nghĩa" của tham lam: `M_k` là **khối lượng lớn nhất có thể** trước khi phải đối mặt với một thiên thạch "lớn" (≥ `a_k`).

**Đúng với phần tử bằng nhau:** điều kiện là `≥`, nên `mass == x` vẫn phá được — code dùng `if mass < x: return False` khớp đúng.

## 11. Phân tích Time / Space Complexity

`n = len(asteroids)`, `V = max(asteroids) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | O(n!·n) | O(n) |
| 1b. Mô phỏng ăn bất kỳ | O(n²) | O(n) |
| 2. Sắp xếp + quét | O(n log n) | O(n) (`sorted`) / O(1) thêm (`sort`) |
| 3. Sắp xếp + dừng sớm | O(n log n), có thể O(n) | O(n) |
| 4. Đếm phân phối | O(n + V) | O(V) |
| 5. Công thức tiền tố | O(n log n) | O(n) |

Khối lượng tối đa: `10⁵ + 10⁵·10⁵ ≈ 10¹⁰` — vượt `int32` (C++/Java cần `long long`/`long`), Python không lo.

In [18]:
rng = random.Random(2126)
big = [rng.randint(1, 10**5) for _ in range(10**5)]
for name, f in [("simulate_any (n=3000)", None), ("sort_scan", sort_scan), ("sort_early", sort_early),
                ("counting", counting), ("prefix_formula", prefix_formula)]:
    if f is None:
        small = big[:3000]
        t = time.perf_counter(); r = simulate_any(10**5, small); dt = time.perf_counter() - t
    else:
        t = time.perf_counter(); r = f(1, big); dt = time.perf_counter() - t
    print(f"{name:<22} -> {r}  {dt*1000:.1f} ms")

simulate_any (n=3000)  -> True  0.5 ms
sort_scan              -> False  15.5 ms
sort_early             -> False  15.4 ms
counting               -> False  3.4 ms
prefix_formula         -> False  15.1 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute": brute, "simulate_any": simulate_any, "sort_scan": sort_scan,
       "SolutionSort": lambda m, a: SolutionSort().asteroidsDestroyed(m, list(a)),
       "sort_early": sort_early, "counting": counting, "prefix_formula": prefix_formula,
       "Solution": Solution().asteroidsDestroyed}
edge = examples + [
    (1, [1], True),              # bằng nhau vẫn phá được
    (1, [2], False),
    (1, [1, 2], True),           # 1 -> 2 -> phá 2
    (1, [1, 3], False),          # 1 -> 2 < 3
    (3, [3, 3, 3], True),
    (1, [1, 1, 3, 6], True),     # 1->2->3->6->12
    (1, [1, 1, 3, 7], False),    # 1->2->3->6 < 7
    (10**5, [10**5] * 5, True),
]
for mass, ast, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and len(ast) > 8: continue
        assert f(mass, list(ast)) == exp, (name, mass, ast, exp)
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên với vét cạn hoán vị (n ≤ 7)

In [20]:
rng = random.Random(42)
cnt_true = 0
for _ in range(1500):
    n = rng.randint(1, 7)
    ast = [rng.randint(1, 30) for _ in range(n)]
    mass = rng.randint(1, 15)
    ref = brute(mass, ast)
    cnt_true += ref
    for name, f in ALL.items():
        a = list(ast)
        assert f(mass, a) == ref, (name, mass, ast)
print("1500 case OK, số case True:", cnt_true)

1500 case OK, số case True: 591


### 12.3 Đối chiếu các cách nhanh trên input lớn

In [21]:
rng = random.Random(7)
for _ in range(30):
    n = rng.randint(1, 3000)
    ast = [rng.randint(1, 10**5) for _ in range(n)]
    mass = rng.choice([1, 100, 5000, 10**5])
    ref = sort_scan(mass, ast)
    assert simulate_any(mass, ast) == ref if n <= 800 else True
    for name in ["sort_early", "counting", "prefix_formula", "Solution"]:
        assert ALL[name](mass, list(ast)) == ref
print("30 case lớn OK")

30 case lớn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp, đi theo thứ tự input

In [22]:
def wrong_no_sort(mass, asteroids):
    for x in asteroids:
        if mass < x: return False
        mass += x
    return True

print("bản sai :", wrong_no_sort(1, [2, 1]))
print("bản đúng:", Solution().asteroidsDestroyed(1, [2, 1]))

bản sai : False
bản đúng: True


**Vì sao sai:** đề cho phép chọn thứ tự; thứ tự input có thể đặt thiên thạch lớn lên trước. **Sửa:** `sorted(asteroids)`.

### Lỗi 2 — Sắp xếp **giảm dần**

In [23]:
def wrong_desc(mass, asteroids):
    for x in sorted(asteroids, reverse=True):
        if mass < x: return False
        mass += x
    return True

print("bản sai :", wrong_desc(10, [3, 9, 19, 5, 21]))
print("bản đúng:", Solution().asteroidsDestroyed(10, [3, 9, 19, 5, 21]))

bản sai : False
bản đúng: True


**Vì sao sai:** gặp 21 ngay đầu với mass 10. Trực giác "đánh trùm trước" sai ở đây vì hành tinh **mạnh lên** sau mỗi lần ăn. **Sửa:** tăng dần.

### Lỗi 3 — Dùng `<=` thay vì `<` (coi bằng nhau là thua)

In [24]:
def wrong_strict(mass, asteroids):
    for x in sorted(asteroids):
        if mass <= x: return False          # SAI: mass == x vẫn phá được
        mass += x
    return True

print("bản sai :", wrong_strict(1, [1]))
print("bản đúng:", Solution().asteroidsDestroyed(1, [1]))

bản sai : False
bản đúng: True


**Sửa:** đề nói "greater than **or equal**" ⇒ thua chỉ khi `mass < x`.

### Lỗi 4 — Chỉ so tổng: `mass + sum(asteroids) - max >= max`

In [25]:
def wrong_total_only(mass, asteroids):
    mx = max(asteroids)
    return mass + sum(asteroids) - mx >= mx

print("bản sai :", wrong_total_only(1, [3, 3, 3]))
print("bản đúng:", Solution().asteroidsDestroyed(1, [3, 3, 3]))

bản sai : True
bản đúng: False


**Vì sao sai:** điều kiện "ăn hết các thiên thạch khác thì đủ lớn để ăn cái lớn nhất" là **cần** nhưng không **đủ** — có thể ngay bước đầu đã không ăn được gì (mass 1 < 3). Ví dụ 2 của đề chỉ minh họa cho thiên thạch cuối, nhưng cản trở có thể ở **bất kỳ** bước nào. **Sửa:** kiểm tra từng bước sau khi sắp xếp.

### Lỗi 5 — `sort()` trả về `None`

In [26]:
def wrong_sort_none(mass, asteroids):
    try:
        for x in asteroids.sort():          # SAI: sort() trả None
            if mass < x: return False
            mass += x
        return True
    except TypeError as e:
        return f"TypeError: {e}"

print("bản sai :", wrong_sort_none(10, [3, 9, 19, 5, 21]))
print("bản đúng:", Solution().asteroidsDestroyed(10, [3, 9, 19, 5, 21]))

bản sai : TypeError: 'NoneType' object is not iterable
bản đúng: True


**Sửa:** `asteroids.sort()` trên một dòng riêng rồi `for x in asteroids`, hoặc `for x in sorted(asteroids)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thứ tự tùy ý" + "mỗi bước làm mình mạnh hơn (không giảm)" ⇒ **tham lam: làm việc dễ nhất trước** ⇒ sắp xếp tăng.
- Chứng minh nhanh trong đầu bằng **đổi chỗ hai phần tử kề nhau** (exchange argument).
- Luôn để ý `>=` hay `>` trong đề.
- Dừng sớm khi trạng thái đã "vô địch" (mass ≥ max).
- Bài cùng họ: 2551/1798 Maximum Number of Consecutive Values You Can Make (tổng tiền tố + sắp xếp), 330 Patching Array, 1029, 2178, "bag of tokens" (948), IPO (502 — tham lam + heap khi có điều kiện vốn).

## 15. Biến thể của bài

### 15.1 Biến thể A — Khối lượng ban đầu **nhỏ nhất** để phá hết

Sau khi sắp xếp, cần `mass ≥ a_k − pre[k]` với mọi `k` ⇒ `mass_min = max(1, max_k (a_k − pre[k]))` (đề ràng buộc mass ≥ 1).

In [27]:
def min_initial_mass(asteroids):
    a = sorted(asteroids); pre = 0; need = 1
    for x in a:
        need = max(need, x - pre)
        pre += x
    return need

def min_initial_mass_brute(asteroids):
    m = 1
    while not Solution().asteroidsDestroyed(m, asteroids): m += 1
    return m

print(min_initial_mass([3, 9, 19, 5, 21]), min_initial_mass([4, 9, 23, 4]))
rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(1, 40) for _ in range(rng.randint(1, 8))]
    assert min_initial_mass(a) == min_initial_mass_brute(a)
print("khớp brute force")

3 6
khớp brute force


### 15.2 Biến thể B — Phá được **tối đa bao nhiêu** thiên thạch (hành tinh không chết vì có thể dừng lại)

Tham lam y hệt: ăn tăng dần đến khi gặp cái không ăn được, đếm số đã ăn. (Lập luận giống mục 10: khi dừng ở `a_k`, không thứ tự nào ăn được thiên thạch nào trong nhóm ≥ `a_k`.)

In [28]:
def max_destroyed(mass, asteroids):
    cnt = 0
    for x in sorted(asteroids):
        if mass < x: break
        mass += x; cnt += 1
    return cnt

def max_destroyed_brute(mass, asteroids):
    best = 0
    for order in permutations(asteroids):
        m, c = mass, 0
        for x in order:
            if m < x: break
            m += x; c += 1
        best = max(best, c)
    return best

print(max_destroyed(5, [4, 9, 23, 4]), max_destroyed(1, [5, 1, 1]))
rng = random.Random(152)
for _ in range(300):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 6))]; m = rng.randint(1, 8)
    assert max_destroyed(m, a) == max_destroyed_brute(m, a)
print("khớp brute force")

3 2
khớp brute force


### 15.3 Biến thể C — Hành tinh chỉ nhận **một nửa** khối lượng (làm tròn xuống)

`mass += x // 2`. Tham lam tăng dần vẫn đúng? Lập luận đổi chỗ: với `y > x` kề nhau, `M ≥ y > x`, sau khi ăn `x`: `M + x//2 ≥ M ≥ y` ✔, và tổng sau cặp vẫn là `M + x//2 + y//2`. Vậy **vẫn đúng** — vì phép cộng thêm luôn không âm và không phụ thuộc thứ tự. Kiểm chứng:

In [29]:
def half_gain(mass, asteroids):
    for x in sorted(asteroids):
        if mass < x: return False
        mass += x // 2
    return True

def half_gain_brute(mass, asteroids):
    for order in permutations(asteroids):
        m = mass; ok = True
        for x in order:
            if m < x: ok = False; break
            m += x // 2
        if ok: return True
    return False

rng = random.Random(153)
for _ in range(400):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 6))]; m = rng.randint(1, 15)
    assert half_gain(m, a) == half_gain_brute(m, a)
print("biến thể nửa khối lượng: tham lam vẫn đúng")

biến thể nửa khối lượng: tham lam vẫn đúng


**Bài học:** tham lam tăng dần đúng khi "phần thưởng" của mỗi thiên thạch **không âm** và **không phụ thuộc** thời điểm ăn. Nếu phần thưởng có thể âm (ví dụ thiên thạch "độc" làm giảm khối lượng) thì cần suy nghĩ lại (giống bài IPO / bag of tokens).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | thử hết thứ tự | O(n!·n) | O(n) | kiểm chứng n ≤ 8 |
| 1b. Ăn bất kỳ | mô phỏng trực tiếp | O(n²) | O(n) | trực quan, n nhỏ |
| 2. Sắp xếp + quét | ăn từ nhỏ tới lớn | O(n log n) | O(n) | **mặc định** |
| 3. + dừng sớm | dừng khi mass ≥ max | O(n log n) / O(n) | O(n) | mass ban đầu lớn |
| 4. Đếm phân phối | counting sort | O(n + V) | O(V) | giá trị nhỏ, ngôn ngữ biên dịch |
| 5. Tiền tố | `all(mass+pre[k] ≥ a[k])` | O(n log n) | O(n) | viết gọn, dễ mở rộng (biến thể A) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `asteroidsDestroyed(2, [1, 2, 4, 8, 17])` và `asteroidsDestroyed(2, [1, 2, 4, 8, 18])`?

**Bài 2 (tính tay):** Với `asteroids = [2, 2, 7, 30]`, khối lượng ban đầu nhỏ nhất để phá hết là bao nhiêu?

**Bài 3 (điền TODO):** Viết `bt3_todo(mass, asteroids)` trả về **thứ tự** (list) phá hết theo tham lam, hoặc `[]` nếu không thể.

**Bài 4 (tự viết lại):** Viết `bt4_todo(mass, asteroids)` dùng **counting sort** (cách 4) không nhìn code.

**Bài 5 (mini challenge):** Được phép **bỏ qua tối đa một** thiên thạch (không cần phá nó). Viết `bt5_todo(mass, asteroids)` trả `True` nếu phá được tất cả những thiên thạch còn lại.

In [30]:
def bt3_todo(mass, asteroids):
    # TODO: trả về thứ tự phá hết, hoặc []
    pass

def bt4_todo(mass, asteroids):
    # TODO: counting sort
    pass

def bt5_todo(mass, asteroids):
    # TODO: được bỏ qua tối đa một thiên thạch
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp xếp sẵn rồi. Mức 2: mass: 2 → 3 → 5 → 9 → 17. Mức 3: 17 ≥ 17 ⇒ `True`; 17 < 18 ⇒ `False`.

**Bài 2.** Mức 1: dùng `a_k − pre[k]`. Mức 2: `2−0 = 2`, `2−2 = 0`, `7−4 = 3`, `30−11 = 19`. Mức 3: đáp án `19`.

**Bài 3.** Mức 1: thứ tự tham lam là `sorted`. Mức 2: chỉ cần kiểm tra nó có thành công không. Mức 3: `a = sorted(...)`; nếu `Solution` trả `True` thì `return a` else `[]`.

**Bài 4.** Mức 1: mảng `cnt` độ dài `max + 1`. Mức 2: duyệt `v` tăng dần, bỏ qua `cnt[v] == 0`. Mức 3: `if mass < v: return False; mass += v * cnt[v]`.

**Bài 5.** Mức 1: nên bỏ qua thiên thạch nào? Mức 2: bỏ qua cái **lớn nhất** là tốt nhất (nó vừa khó nhất, vừa chỉ được ăn ở cuối). Mức 3: sắp xếp, bỏ phần tử cuối, chạy tham lam trên `n − 1` phần tử đầu (nếu n = 1 thì `True`).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `True`, `False`. **Bài 2:** `19`.

In [31]:
assert Solution().asteroidsDestroyed(2, [1, 2, 4, 8, 17]) is True
assert Solution().asteroidsDestroyed(2, [1, 2, 4, 8, 18]) is False
assert min_initial_mass([2, 2, 7, 30]) == 19

def bt3(mass, asteroids):
    a = sorted(asteroids)
    return a if Solution().asteroidsDestroyed(mass, a) else []

def bt4(mass, asteroids):
    V = max(asteroids); cnt = [0] * (V + 1)
    for x in asteroids: cnt[x] += 1
    for v in range(1, V + 1):
        if cnt[v]:
            if mass < v: return False
            mass += v * cnt[v]
    return True

def bt5(mass, asteroids):
    a = sorted(asteroids)
    return Solution().asteroidsDestroyed(mass, a[:-1])

def bt5_brute(mass, asteroids):
    if Solution().asteroidsDestroyed(mass, asteroids): return True
    return any(Solution().asteroidsDestroyed(mass, asteroids[:i] + asteroids[i + 1:]) for i in range(len(asteroids)))

assert bt3(10, [3, 9, 19, 5, 21]) == [3, 5, 9, 19, 21] and bt3(5, [4, 9, 23, 4]) == []
assert bt5(5, [4, 9, 23, 4]) is True and bt5(1, [2, 2]) is False
rng = random.Random(1926)
for _ in range(2000):
    a = [rng.randint(1, 30) for _ in range(rng.randint(1, 7))]; m = rng.randint(1, 15)
    ref = brute(m, a) if len(a) <= 6 else sort_scan(m, a)
    assert bt4(m, a) == ref
    order = bt3(m, a)
    assert (order != []) == ref and (not order or play(m, order))
    assert bt5(m, a) == bt5_brute(m, a)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: vì sao bỏ thiên thạch lớn nhất là tối ưu? Nếu bỏ một thiên thạch `y` khác mà thành công, thì trong dãy còn lại có thiên thạch lớn nhất `z ≥ y`. Thay việc bỏ `y` bằng bỏ `z`: dãy mới là dãy cũ với `z` thay bằng `y` (nhỏ hơn hoặc bằng) — từng bước so sánh chỉ dễ hơn hoặc bằng, nên vẫn thành công.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for x in sorted(asteroids):   # tham lam: dễ trước, khó sau
    if mass < x:              # đúng dấu: bằng nhau vẫn phá được
        return False
    mass += x
return True
```

**Checklist:**
- [ ] Sắp xếp **tăng dần**.
- [ ] Điều kiện thua là `mass < x` (không phải `<=`).
- [ ] Không viết `for x in asteroids.sort()`.
- [ ] (C++/Java) dùng kiểu 64-bit cho `mass`.
- [ ] Đã thử: `mass == x`, thiên thạch lớn đứng đầu input, ví dụ cản trở ở bước đầu (`1, [3,3,3]`).